# 实验 12 — dlt 自动展开嵌套 JSON

**目标**：看 dlt 把 REST API 返回的嵌套结构自动拍平成关系表 —— 这是 dlt 相对手写 ingestion 最有竞争力的地方。

## 两种自动行为

| 结构 | dlt 处理方式 |
|---|---|
| 嵌套 dict（1:1） | 拍平成父表上的列，加前缀（`address__city`、`address__geo__lat`）|
| 嵌套 list（1:N） | 拆出 **child table**，自动加外键列 `_dlt_parent_id`、`_dlt_root_id`、`_dlt_list_idx` |

本实验用 [dlt_pipelines/users_nested.py](../dlt_pipelines/users_nested.py)，源是 JSONPlaceholder：每个 user 有嵌套 address（dict）和 nested posts 列表。

In [ ]:
import subprocess
r = subprocess.run(['uv','run','python','dlt_pipelines/users_nested.py'],
                   capture_output=True, text=True, cwd='..')
print(r.stdout[-1500:])
if r.returncode != 0: print('STDERR:', r.stderr[-500:])

In [ ]:
import duckdb
con = duckdb.connect('../data/sandbox.duckdb')
print('=== raw_users 里 dlt 建了哪些表？===')
con.execute('show tables from raw_users').df()

In [ ]:
# users 父表：注意嵌套 dict 被拍平成 address__street、address__geo__lat 这种列
con.execute('describe raw_users.users').df()

In [ ]:
# 嵌套 list 'posts' 被拆成了 users__posts 子表
con.execute('describe raw_users.users__posts').df()

In [ ]:
# 父子表通过 _dlt_id <-> _dlt_parent_id join
con.execute("""
    select u.id as user_id, u.name, p.post_id, p.title
    from raw_users.users u
    join raw_users.users__posts p on p._dlt_parent_id = u._dlt_id
    order by u.id, p._dlt_list_idx
    limit 10
""").df()

In [ ]:
# _dlt_root_id 在多层嵌套时也保留了顶层根的指针，便于做 row-level lineage
con.execute("""
    select column_name, data_type
    from information_schema.columns
    where table_schema = 'raw_users' and table_name = 'users__posts'
      and column_name like '\_dlt%' escape '\\'
""").df()

## 生产环境踩坑提示

- **命名爆炸**：深嵌套（user.profile.address.geo.coordinates.lat）会产出 `user__profile__address__geo__coordinates__lat` 这种长列名，超过 Postgres 63 字符限制。`.dlt/config.toml` 里可以设 `[normalize.json_normalizer.config] max_nesting=N` 来截断深度，剩下的会以 JSON blob 存。
- **child table 主键**：dlt 自动给每行一个 `_dlt_id`。如果业务希望用 `post_id` 做主键，要在 resource 上声明 `primary_key`。
- **可空字段 → 列消失**：如果某次 load 里 nested field 全是 null，dlt 可能不建对应列，下次又有数据时再建。下游 dbt 写 select 列名时容易踩坑。
- **大 JSON 字段**：超过 column 大小限制的会被 dlt 存为 JSON blob 类型（DuckDB 是 JSON）。`SELECT json_extract(...)` 可以再剥。

## 思考题

- 在 dbt 里写一个 `stg_user_posts.sql`，从 `raw_users.users__posts` 选 post_id、title，并 join `raw_users.users` 拿用户名。
- 如果你不想 dlt 自动拆 list（比如希望 posts 留在父表里以 JSON 形式存），怎么做？（提示：`columns={'posts': {'data_type': 'json'}}`）
- `max_nesting` 设为 1 之后再重跑，address.geo 会变成什么？